# 🚗 UNLET-ADAS: Low-Light Video Enhancement
### B.E. Major Project | SJBIT Bengaluru | CSE 2025-26
**GitHub:** https://github.com/DEEK-SHITH/UNLET-ADAS

---

| Cell | What it does |
|---|---|
| 1 | Setup — install packages, mount Drive, clone GitHub |
| 2 | Configuration — all paths in one place |
| 3 | Prepare extra low-light training data (optional) |
| 4 | Build model |
| 5 | Train model (100 epochs, ~60 min) |
| 6 | Load best weights + color check |

Cell 3 is optional but recommended: the paper's own conclusion names
LOL's 485 pairs as mostly indoor/urban, so it folds in extra
*unpaired* low-light images (no ground truth needed — the loss
already runs unsupervised-only for any sample without one) from this
repo's own real night-drive footage, and optionally the ExDark
dataset, to generalize the enhancement model further.

In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================

# Anti-disconnect — run this first
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var btns = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<btns.length;i++){
        if(btns[i].id=="connect") btns[i].click();
    }
}
setInterval(ClickConnect, 55000)
'''))
print('Anti-disconnect active!')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install ultralytics scikit-image timm gdown -q

# Clone or update GitHub repo
import os
if not os.path.exists('/content/UNLET-ADAS'):
    !git clone https://github.com/DEEK-SHITH/UNLET-ADAS.git /content/UNLET-ADAS
    print('Repo cloned!')
else:
    !cd /content/UNLET-ADAS && git pull
    print('Repo updated!')

# Add to Python path
import sys
if '/content/UNLET-ADAS' not in sys.path:
    sys.path.insert(0, '/content/UNLET-ADAS')

# Imports
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json
import time
import shutil
import warnings
warnings.filterwarnings('ignore')
from glob import glob
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity  as calc_ssim

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('Setup complete!')

In [ ]:
# ============================================================
# CELL 2 — Configuration
# All paths are defined here. Edit only this cell if paths change.
# ============================================================

SAVE_DIR    = '/content/drive/MyDrive/UNLET_Project/checkpoints'
OUTPUT_DIR  = '/content/drive/MyDrive/UNLET_Project/results'
VIDEO_INPUT = '/content/drive/MyDrive/UNLET_Project/night_drive.mp4'
LOL_ROOT    = '/content/drive/MyDrive/UNLET_Project/lol_dataset'
LOCAL_LOL   = '/content/lol_dataset'
WEIGHTS     = os.path.join(SAVE_DIR, 'zerodce_cbam_best.pt')
IMAGE_SIZE  = 256

# Extra UNPAIRED low-light training data (Cell 3), beyond LOL's
# 485 pairs -- see that cell's comment for why. Both default on;
# ExDark is a ~1.5GB download so the first run will take longer.
EXTRA_LOWLIGHT_DIR   = '/content/extra_lowlight'
USE_NIGHT_DRIVE_FRAMES = True
USE_EXDARK              = True

# If training was interrupted last time (Colab disconnect, GPU quota
# runout, closed tab), this picks back up from the last completed
# epoch instead of starting over -- reads resume_state.pt (saved
# after every epoch) from SAVE_DIR. Harmless no-op if none exists yet.
RESUME = True

os.makedirs(SAVE_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Checking paths...')
checks = {
    'Video input' : VIDEO_INPUT,
    'LOL train'   : os.path.join(LOL_ROOT, 'our485', 'low'),
    'LOL val'     : os.path.join(LOL_ROOT, 'eval15', 'low'),
    'Save dir'    : SAVE_DIR,
    'Output dir'  : OUTPUT_DIR,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    info = (f'{len(os.listdir(path))} files'
            if exists and os.path.isdir(path)
            else 'found' if exists else 'MISSING')
    print(f'  {"OK" if exists else "MISSING"} | {label}: {info}')

if all_ok:
    print('\nAll paths OK! Ready to proceed.')
else:
    print('\nFix MISSING paths before continuing.')

In [ ]:
# ============================================================
# CELL 3 — Prepare Extra Low-Light Training Data (optional)
# LOL's 485 pairs are mostly indoor/urban (the paper's own conclusion
# names this as a generalization gap). This folds in extra UNPAIRED
# low-light images -- no ground truth needed, since UNLETLoss already
# falls back to its unsupervised losses (color constancy / exposure /
# spatial consistency / smoothness) for any sample without one:
#   1. Real frames from this repo's own night-drive footage
#      (results/original_night_drive.mp4) -- zero extra download.
#   2. Optionally, the official ExDark dataset (~7,363 images across
#      10 real-world lighting conditions) -- reuses the same
#      Google-Drive downloader built for the low-light detector.
# Set USE_NIGHT_DRIVE_FRAMES / USE_EXDARK to False in Cell 2 to skip
# either source and train on LOL alone, as before.
# ============================================================

from src.prepare_extra_lowlight import build_extra_lowlight_set

EXTRA_LOW_DIRS = []
if USE_NIGHT_DRIVE_FRAMES or USE_EXDARK:
    video_paths = (
        [os.path.join('/content/UNLET-ADAS', 'results',
                       'original_night_drive.mp4')]
        if USE_NIGHT_DRIVE_FRAMES else [])
    EXTRA_LOW_DIRS = build_extra_lowlight_set(
        EXTRA_LOWLIGHT_DIR,
        video_paths=video_paths,
        include_exdark=USE_EXDARK)

print(f'\nExtra low-light dirs ready: {EXTRA_LOW_DIRS}')


In [ ]:
# ============================================================
# CELL 4 — Build Model + Loss Functions
# ============================================================

# ── Model ────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, ch, ratio=8):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(ch, max(ch//ratio,1), bias=False),
            nn.ReLU(),
            nn.Linear(max(ch//ratio,1), ch, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        B,C,_,_ = x.shape
        a = self.fc(self.avg(x).view(B,C))
        m = self.fc(self.mx(x).view(B,C))
        return self.sig(a+m).view(B,C,1,1) * x

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2,1,7,padding=3,bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(1,keepdim=True)
        mx, _ = x.max(1,keepdim=True)
        return self.sig(self.conv(torch.cat([avg,mx],1))) * x

class CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.ca = ChannelAttention(ch)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

def dw_block(ic, oc):
    return nn.Sequential(
        nn.Conv2d(ic,ic,3,padding=1,groups=ic,bias=False),
        nn.Conv2d(ic,oc,1,bias=False),
        nn.BatchNorm2d(oc),
        nn.ReLU(inplace=True))

class ZeroDCECBAM(nn.Module):
    def __init__(self, iters=8, ch=32):
        super().__init__()
        self.iters = iters
        self.e1=dw_block(3, ch);  self.cb1=CBAM(ch)
        self.e2=dw_block(ch,ch);  self.cb2=CBAM(ch)
        self.e3=dw_block(ch,ch);  self.cb3=CBAM(ch)
        self.e4=dw_block(ch,ch);  self.cb4=CBAM(ch)
        self.d3=dw_block(ch*2,ch);self.cb5=CBAM(ch)
        self.d2=dw_block(ch*2,ch);self.cb6=CBAM(ch)
        self.d1=dw_block(ch*2,ch);self.cb7=CBAM(ch)
        self.out=nn.Sequential(
            nn.Conv2d(ch,3*iters,3,padding=1,bias=False),
            nn.Tanh())
    def forward(self, x):
        e1=self.cb1(self.e1(x))
        e2=self.cb2(self.e2(e1))
        e3=self.cb3(self.e3(e2))
        e4=self.cb4(self.e4(e3))
        d3=self.cb5(self.d3(torch.cat([e4,e3],1)))
        d2=self.cb6(self.d2(torch.cat([d3,e2],1)))
        d1=self.cb7(self.d1(torch.cat([d2,e1],1)))
        curves = self.out(d1)
        enh = x
        for i in range(self.iters):
            A = curves[:,i*3:(i+1)*3]
            enh = torch.clamp(enh + A*enh*(1-enh), 0, 1)
        return enh, curves

model = ZeroDCECBAM(iters=8, ch=32).to(DEVICE)
params = sum(p.numel() for p in model.parameters())

# Verify model
with torch.no_grad():
    t = torch.rand(2,3,256,256).to(DEVICE)
    e, c = model(t)
print(f'Model parameters : {params:,}')
print(f'Input shape      : {t.shape}')
print(f'Output shape     : {e.shape}')
print(f'Curves shape     : {c.shape}')

# ── Loss Functions ───────────────────────────────────────────
class VGGPerceptual(nn.Module):
    def __init__(self):
        super().__init__()
        v = tvm.vgg19(weights=tvm.VGG19_Weights.IMAGENET1K_V1).features
        self.s1=nn.Sequential(*list(v)[:4]).eval()
        self.s2=nn.Sequential(*list(v)[4:9]).eval()
        self.s3=nn.Sequential(*list(v)[9:18]).eval()
        for p in self.parameters(): p.requires_grad_(False)
        self.register_buffer('mean',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(DEVICE))
        self.register_buffer('std', torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(DEVICE))
    def forward(self, x, y):
        x=(x-self.mean)/self.std; y=(y-self.mean)/self.std
        loss=0
        for sl in [self.s1,self.s2,self.s3]:
            x=sl(x); y=sl(y); loss+=F.l1_loss(x,y)
        return loss

perc_fn = VGGPerceptual().to(DEVICE)

def color_loss(x):
    m=x.mean(dim=[2,3])
    r,g,b=m[:,0],m[:,1],m[:,2]
    return ((r-g)**2+(r-b)**2+(g-b)**2).mean()

def expo_loss(x, tgt=0.55):
    gray=(0.299*x[:,0]+0.587*x[:,1]+0.114*x[:,2]).unsqueeze(1)
    return F.mse_loss(F.avg_pool2d(gray,16,16),
                      torch.ones_like(F.avg_pool2d(gray,16,16))*tgt)

def smooth_loss(c):
    return (c[:,:,:,1:]-c[:,:,:,:-1]).abs().mean()+(c[:,:,1:,:]-c[:,:,:-1,:]).abs().mean()

def spatial_loss(enh, orig):
    return F.mse_loss(F.avg_pool2d(enh,4,4),F.avg_pool2d(orig,4,4))

def total_loss(enh, curves, orig, norm=None):
    # color_loss weight=100 prevents green/purple tint
    loss = (100.0*color_loss(enh) +
             10.0*expo_loss(enh) +
              1.0*spatial_loss(enh,orig) +
            200.0*smooth_loss(curves))
    # Per-sample mask, not a whole-batch norm.sum() check: a batch can
    # mix LOL pairs with extra unpaired low-light images (all-zero
    # norm rows), and a whole-batch check would wrongly apply the
    # supervised loss to those unpaired rows too.
    if norm is not None:
        has_gt = norm.reshape(norm.shape[0],-1).sum(dim=1) > 0
        if has_gt.any():
            e, n = enh[has_gt], norm[has_gt]
            loss = loss + (1.0*F.l1_loss(e,n) +
                           0.1*perc_fn(e,n) +
                           2.0*(1-F.cosine_similarity(
                               e.reshape(e.shape[0],-1),
                               n.reshape(n.shape[0],-1)).mean()))
    return loss

# Verify loss
with torch.no_grad():
    d=torch.rand(2,3,256,256).to(DEVICE)
    ev,cv=model(d)
    lv=total_loss(ev,cv,d,d)
print(f'Loss check       : {float(lv):.4f}')
print(f'Color loss weight: 100.0 (prevents color tint)')
print('Model and loss functions ready!')

In [ ]:
# ============================================================
# CELL 5 — Load Dataset and Train
# Takes approximately 55-65 minutes on T4 GPU (a bit more if
# extra low-light data from Cell 3 was included)
# ============================================================

# ── Dataset ──────────────────────────────────────────────────
def _glob_images(d):
    return sorted(glob(os.path.join(d,'*.png'))+glob(os.path.join(d,'*.jpg'))+
                  glob(os.path.join(d,'*.jpeg')))

class LOLDataset(Dataset):
    def __init__(self, low_dir, high_dir=None, size=256, extra_low_dirs=None):
        self.size = size
        self.lows = _glob_images(low_dir)
        self.hmap = {}
        if high_dir:
            highs = _glob_images(high_dir)
            self.hmap = {os.path.basename(p):p for p in highs}
        print(f'  {len(self.lows)} paired images from {os.path.basename(low_dir)}')
        for extra_dir in (extra_low_dirs or []):
            if not extra_dir or not os.path.isdir(extra_dir):
                print(f'  WARNING: extra_low_dir not found, skipping: {extra_dir}')
                continue
            extra_imgs = [p for p in _glob_images(extra_dir)
                          if os.path.basename(p) not in self.hmap]
            self.lows.extend(extra_imgs)
            print(f'  {len(extra_imgs)} unpaired images from '
                  f'{os.path.basename(extra_dir.rstrip("/"))} (unsupervised losses only)')
    def _t(self, path):
        img=Image.open(path).convert('RGB').resize((self.size,self.size),Image.BICUBIC)
        return torch.from_numpy(np.array(img,dtype=np.float32)/255.0).permute(2,0,1)
    def __len__(self): return len(self.lows)
    def __getitem__(self, i):
        lp=self.lows[i]; low=self._t(lp)
        name=os.path.basename(lp)
        high=self._t(self.hmap[name]) if name in self.hmap else torch.zeros_like(low)
        if torch.rand(1)>0.5: low=torch.flip(low,[-1]); high=torch.flip(high,[-1])
        if torch.rand(1)>0.5: low=torch.flip(low,[-2]); high=torch.flip(high,[-2])
        k=int(torch.randint(0,4,(1,)))
        return torch.rot90(low,k,[-2,-1]),torch.rot90(high,k,[-2,-1])

# Copy dataset to local storage for faster training
if not os.path.exists(f'{LOCAL_LOL}/our485/low') or \
   len(os.listdir(f'{LOCAL_LOL}/our485/low'))==0:
    print('Copying dataset to local storage...')
    for split in ['our485/low','our485/high','eval15/low','eval15/high']:
        src=os.path.join(LOL_ROOT,split)
        dst=os.path.join(LOCAL_LOL,split)
        os.makedirs(dst,exist_ok=True)
        shutil.copytree(src,dst,dirs_exist_ok=True)
        print(f'  {split}: {len(os.listdir(dst))} images')
else:
    print('Dataset already in local storage')

print('\nLoading datasets...')
train_ds=LOLDataset(f'{LOCAL_LOL}/our485/low',f'{LOCAL_LOL}/our485/high',
                    extra_low_dirs=EXTRA_LOW_DIRS)
val_ds  =LOLDataset(f'{LOCAL_LOL}/eval15/low', f'{LOCAL_LOL}/eval15/high')
train_dl=DataLoader(train_ds,batch_size=8,shuffle=True,num_workers=2,pin_memory=True)
val_dl  =DataLoader(val_ds,  batch_size=4,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

# ── Training ─────────────────────────────────────────────────
EPOCHS      = 100
PATIENCE    = 20
opt         = torch.optim.Adam(model.parameters(),lr=2e-4,weight_decay=1e-5)
sched       = CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
best_val    = float('inf')
patience    = 0
history     = {'train':[],'val':[],'psnr':[],'ssim':[]}
start_epoch = 0
RESUME_PATH = os.path.join(SAVE_DIR, 'resume_state.pt')

if RESUME and os.path.exists(RESUME_PATH):
    ckpt = torch.load(RESUME_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['optimizer'])
    sched.load_state_dict(ckpt['scheduler'])
    best_val    = ckpt['best_val']
    patience    = ckpt['patience']
    history     = ckpt['history']
    start_epoch = ckpt['epoch'] + 1
    if ckpt.get('total_epochs') != EPOCHS:
        print(f'WARNING: this checkpoint was started with EPOCHS='
              f'{ckpt.get("total_epochs")}, but EPOCHS={EPOCHS} now -- '
              'set them equal for a clean cosine LR schedule.')
    print(f'Resuming from epoch {start_epoch + 1} '
          f'(best val so far: {best_val:.4f})')

print(f'\nStarting training: {EPOCHS} epochs')
print(f'Color loss weight: 100.0 — prevents color tint')
print('-'*65)
t0 = time.time()

for epoch in range(start_epoch, EPOCHS):
    model.train()
    tl=[]
    for low,high in train_dl:
        low,high=low.to(DEVICE),high.to(DEVICE)
        opt.zero_grad()
        enh,curves=model(low)
        norm=high if high.sum()>0 else None
        loss=total_loss(enh,curves,low,norm)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        tl.append(loss.item())
    sched.step()

    model.eval()
    vl,vp,vs=[],[],[]
    with torch.no_grad():
        for low,high in val_dl:
            low,high=low.to(DEVICE),high.to(DEVICE)
            enh,curves=model(low)
            norm=high if high.sum()>0 else None
            vl.append(total_loss(enh,curves,low,norm).item())
            if high.sum()>0:
                en=enh.permute(0,2,3,1).cpu().numpy().clip(0,1)
                hn=high.permute(0,2,3,1).cpu().numpy().clip(0,1)
                for e,h in zip(en,hn):
                    vp.append(calc_psnr(h,e,data_range=1.0))
                    vs.append(calc_ssim(h,e,channel_axis=2,data_range=1.0))

    tl_m=float(np.mean(tl)); vl_m=float(np.mean(vl))
    vp_m=float(np.mean(vp)) if vp else 0.0
    vs_m=float(np.mean(vs)) if vs else 0.0
    elapsed=(time.time()-t0)/60
    history['train'].append(tl_m); history['val'].append(vl_m)
    history['psnr'].append(vp_m);  history['ssim'].append(vs_m)

    if vl_m<best_val:
        best_val=vl_m; patience=0
        torch.save(model.state_dict(),WEIGHTS)
        marker=f'  SAVED  PSNR={vp_m:.2f}dB SSIM={vs_m:.4f}'
    else:
        patience+=1
        marker=f'  patience {patience}/{PATIENCE}'

    print(f'Ep {epoch+1:3d}/{EPOCHS} | '
          f'loss={tl_m:.3f} | val={vl_m:.3f} | '
          f'{elapsed:.1f}m{marker}')

    # Saved every epoch (not just on a new best) so a disconnect or
    # GPU-quota interruption never loses more than one epoch's worth
    # of progress -- set RESUME=True in Cell 2 to pick back up here.
    torch.save({
        'model': model.state_dict(), 'optimizer': opt.state_dict(),
        'scheduler': sched.state_dict(), 'epoch': epoch,
        'best_val': best_val, 'patience': patience, 'history': history,
        'total_epochs': EPOCHS,
    }, RESUME_PATH)

    if patience>=PATIENCE:
        print(f'Early stopping at epoch {epoch+1}'); break

with open(os.path.join(SAVE_DIR,'history.json'),'w') as f:
    json.dump(history,f,indent=2)
print(f'\nTraining done!')
print(f'Best PSNR : {max(history["psnr"]):.2f} dB')
print(f'Best SSIM : {max(history["ssim"]):.4f}')
print(f'Weights   : {WEIGHTS}')

In [ ]:
# ============================================================
# CELL 6 — Load Best Weights + Color Check
# ============================================================

model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()
print('Best weights loaded!')

# Color check on LOL val images
print('\nColor check on enhanced images:')
print('-'*45)
test_imgs = glob(f'{LOCAL_LOL}/eval15/low/*.png')[:5]
all_neutral = True
for img_path in test_imgs:
    img = Image.open(img_path).convert('RGB').resize((256,256))
    arr = np.array(img,dtype=np.float32)/255.0
    t   = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        enh,_ = model(t)
    enh_np = (enh[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
    r=enh_np[:,:,0].mean(); g=enh_np[:,:,1].mean(); b=enh_np[:,:,2].mean()
    diff=max(abs(r-g),abs(r-b),abs(g-b))
    status='NEUTRAL' if diff<15 else 'SLIGHT CAST' if diff<25 else 'COLOR CAST'
    if diff>=25: all_neutral=False
    print(f'  {os.path.basename(img_path)}: R={r:.0f} G={g:.0f} B={b:.0f} → {status}')

print()
if all_neutral:
    print('Colors look natural! Ready to process video.')
else:
    print('Some color cast detected but acceptable for demo.')

In [ ]:
# ============================================================
# CELL 7 — Test on LOL Images
# Shows: Original | AutoContrast | UNLET Enhanced
# ============================================================

def compare_plot(img_path, save_path=None):
    orig = Image.open(img_path).convert('RGB').resize((256,256))
    auto = ImageOps.autocontrast(orig)
    arr  = np.array(orig,dtype=np.float32)/255.0
    t    = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        enh,_ = model(t)
    enh_np  = (enh[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
    enh_pil = Image.fromarray(enh_np)

    ob=np.array(orig).mean()/255
    ab=np.array(auto).mean()/255
    eb=enh_np.mean()/255

    fig,axs=plt.subplots(1,3,figsize=(15,5))
    fig.patch.set_facecolor('#0a0f1e')
    for ax,(img,title,bright,color) in zip(axs,[
        (orig,    f'Original\nBrightness:{ob:.3f}',    ob,'#94a3b8'),
        (auto,    f'AutoContrast\nBrightness:{ab:.3f}',ab,'#f59e0b'),
        (enh_pil, f'UNLET Enhanced\nBrightness:{eb:.3f}',eb,'#22c55e')]):
        ax.imshow(img)
        ax.set_title(title,color=color,fontsize=12,fontweight='bold',pad=8)
        ax.axis('off')
    fig.suptitle(f'UNLET-ADAS — {os.path.basename(img_path)}',
                 color='white',fontsize=13,y=1.02)
    plt.tight_layout(pad=0.5)
    if save_path:
        plt.savefig(save_path,dpi=150,bbox_inches='tight',facecolor='#0a0f1e')
    plt.show(); plt.close()

img_results_dir = os.path.join(OUTPUT_DIR,'image_results')
os.makedirs(img_results_dir, exist_ok=True)
print(f'Testing on {len(test_imgs)} validation images...\n')
for i,p in enumerate(test_imgs):
    compare_plot(p, save_path=os.path.join(img_results_dir,f'result_{i+1:02d}.png'))
print(f'Results saved to: {img_results_dir}')

In [ ]:
# ============================================================
# CELL 8 — Evaluate PSNR and SSIM
# ============================================================

def evaluate(model, low_dir, high_dir):
    lows  = sorted(glob(os.path.join(low_dir, '*.png')))
    highs = sorted(glob(os.path.join(high_dir,'*.png')))
    hmap  = {os.path.basename(p):p for p in highs}
    p_orig,s_orig=[],[]; p_auto,s_auto=[],[]; p_enh,s_enh=[],[]

    for lp in lows:
        name=os.path.basename(lp)
        if name not in hmap: continue
        low_np  = np.array(Image.open(lp).convert('RGB').resize((256,256)),dtype=np.float32)/255.0
        high_np = np.array(Image.open(hmap[name]).convert('RGB').resize((256,256)),dtype=np.float32)/255.0
        auto_np = np.array(ImageOps.autocontrast(Image.open(lp).convert('RGB').resize((256,256))),dtype=np.float32)/255.0
        t   = torch.from_numpy(low_np).permute(2,0,1).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): enh,_=model(t)
        enh_np=enh[0].permute(1,2,0).cpu().numpy().clip(0,1)
        p_orig.append(calc_psnr(high_np,low_np, data_range=1.0))
        p_auto.append(calc_psnr(high_np,auto_np,data_range=1.0))
        p_enh.append( calc_psnr(high_np,enh_np, data_range=1.0))
        s_orig.append(calc_ssim(high_np,low_np, channel_axis=2,data_range=1.0))
        s_auto.append(calc_ssim(high_np,auto_np,channel_axis=2,data_range=1.0))
        s_enh.append( calc_ssim(high_np,enh_np, channel_axis=2,data_range=1.0))

    return {
        'Original'    :{'psnr':float(np.mean(p_orig)),'ssim':float(np.mean(s_orig))},
        'AutoContrast':{'psnr':float(np.mean(p_auto)),'ssim':float(np.mean(s_auto))},
        'UNLET (Ours)':{'psnr':float(np.mean(p_enh)), 'ssim':float(np.mean(s_enh))},
    }

print('Evaluating on LOL eval15...')
results = evaluate(model,f'{LOCAL_LOL}/eval15/low',f'{LOCAL_LOL}/eval15/high')

print('\n'+'='*50)
print('  EVALUATION RESULTS — LOL eval15')
print('='*50)
print(f'  {"Method":<16}  {"PSNR(dB)":>9}  {"SSIM":>8}')
print('-'*50)
for method,scores in results.items():
    m=' <- Our Model' if 'UNLET' in method else ''
    print(f'  {method:<16}  {scores["psnr"]:>9.2f}  {scores["ssim"]:>8.4f}{m}')
print('='*50)

with open(os.path.join(OUTPUT_DIR,'eval_results.json'),'w') as f:
    json.dump(results,f,indent=2)

# Bar chart
methods=[m for m in results]; psnrs=[results[m]['psnr'] for m in methods]
ssims=[results[m]['ssim'] for m in methods]; colors=['#94a3b8','#f59e0b','#22c55e']
fig,axes=plt.subplots(1,2,figsize=(13,5)); fig.patch.set_facecolor('#0f172a')
for ax,vals,ylabel,title in zip(axes,[psnrs,ssims],
    ['PSNR (dB)','SSIM'],['PSNR Comparison','SSIM Comparison']):
    ax.set_facecolor('#1e293b')
    bars=ax.bar(methods,vals,color=colors,width=0.5,edgecolor='white')
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+max(vals)*0.01,
                f'{v:.3f}',ha='center',color='white',fontsize=12,fontweight='bold')
    ax.set_title(title,color='white',fontsize=12,fontweight='bold')
    ax.set_ylabel(ylabel,color='white'); ax.tick_params(colors='white')
    ax.spines[['top','right','left','bottom']].set_color('#334155')
    ax.set_ylim(0,max(vals)*1.18)
fig.suptitle('UNLET-ADAS vs Baselines',color='white',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'metric_comparison.png'),
            dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print('Evaluation complete! Chart saved.')

In [ ]:
# ============================================================
# CELL 9 — Enhance Night Driving Video
# Output 1: enhanced_final.mp4  — enhanced frames only
# Output 2: comparison_final.mp4 — side by side
# ============================================================

ENH_OUTPUT = os.path.join(OUTPUT_DIR,'enhanced_final.mp4')
CMP_OUTPUT = os.path.join(OUTPUT_DIR,'comparison_final.mp4')
SIZE       = 384   # process at 384x384 for quality
BATCH      = 4

if not os.path.exists(VIDEO_INPUT):
    print(f'Video not found: {VIDEO_INPUT}')
    print('Upload night_drive.mp4 to your Drive first')
else:
    cap    = cv2.VideoCapture(VIDEO_INPUT)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30
    W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    enh_w  = cv2.VideoWriter(ENH_OUTPUT, fourcc, fps, (W,H))
    cmp_w  = cv2.VideoWriter(CMP_OUTPUT, fourcc, fps, (W*2,H))

    print(f'Input  : {W}x{H} @ {fps:.0f}fps | {total} frames')
    print(f'Process: {SIZE}x{SIZE} per frame')
    print(f'Output : {CMP_OUTPUT}')
    print('-'*50)

    fb,ob,count=[],[],0
    t0=time.time()

    def proc(fb,ob):
        arr=np.stack(fb).astype(np.float32)/255.0
        t=torch.from_numpy(arr).permute(0,3,1,2).to(DEVICE)
        with torch.no_grad(): enh,_=model(t)
        enp=(enh.permute(0,2,3,1).cpu().numpy()*255).clip(0,255).astype(np.uint8)
        for orig_bgr,enh_rgb in zip(ob,enp):
            ef=cv2.resize(enh_rgb,(W,H))
            eb=cv2.cvtColor(ef,cv2.COLOR_RGB2BGR)
            cv2.rectangle(orig_bgr,(0,0),(220,50),(0,0,0),-1)
            cv2.putText(orig_bgr,'ORIGINAL',(8,35),
                cv2.FONT_HERSHEY_DUPLEX,1.0,(80,80,255),2,cv2.LINE_AA)
            cv2.rectangle(eb,(0,0),(310,50),(0,0,0),-1)
            cv2.putText(eb,'UNLET ENHANCED',(8,35),
                cv2.FONT_HERSHEY_DUPLEX,1.0,(50,220,80),2,cv2.LINE_AA)
            enh_w.write(eb)
            cmp=np.hstack([orig_bgr,eb])
            cv2.line(cmp,(W,0),(W,H),(255,255,255),3)
            cmp_w.write(cmp)

    while True:
        ret,frm=cap.read()
        if not ret: break
        rgb=cv2.cvtColor(frm,cv2.COLOR_BGR2RGB)
        fb.append(cv2.resize(rgb,(SIZE,SIZE))); ob.append(frm.copy()); count+=1
        if len(fb)>=BATCH:
            proc(fb,ob); fb.clear(); ob.clear()
        if count%60==0:
            el=time.time()-t0; eta=el/count*(total-count)
            print(f'  Frame {count:4d}/{total} ({100*count//total}%)  ETA:{eta:.0f}s')

    if fb: proc(fb,ob)
    cap.release(); enh_w.release(); cmp_w.release()

    el=time.time()-t0
    print(f'\nDone! {count} frames in {el/60:.1f} min')
    for p in [ENH_OUTPUT,CMP_OUTPUT]:
        sz=os.path.getsize(p)/(1024*1024)
        print(f'  {os.path.basename(p)}: {sz:.1f} MB')

In [ ]:
# ============================================================
# CELL 10 — Show Video Frame Comparison
# ============================================================

def show_video_comparison(video_path, n=5, save_path=None):
    if not os.path.exists(video_path):
        print(f'Video not found: {video_path}'); return
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    Wh    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))//2
    idxs  = np.linspace(int(total*0.05),int(total*0.95),n,dtype=int)
    fig   = plt.figure(figsize=(n*4,9))
    fig.patch.set_facecolor('#0a0f1e')
    fig.suptitle('UNLET-ADAS: Night Driving Enhancement\nOriginal (top) vs Enhanced (bottom)',
                 color='white',fontsize=13,fontweight='bold',y=1.02)
    gs=gridspec.GridSpec(2,n,figure=fig,hspace=0.06,wspace=0.04)
    for col,idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx))
        ret,frame=cap.read()
        if not ret: continue
        orig=cv2.cvtColor(frame[:,:Wh],cv2.COLOR_BGR2RGB)
        enh =cv2.cvtColor(frame[:,Wh:],cv2.COLOR_BGR2RGB)
        ax0=fig.add_subplot(gs[0,col])
        ax0.imshow(orig); ax0.axis('off')
        ax0.set_title(f'Frame {idx}',color='#64748b',fontsize=8)
        if col==0: ax0.set_ylabel('Original',color='#ef4444',fontsize=11,fontweight='bold',rotation=90)
        ax1=fig.add_subplot(gs[1,col])
        ax1.imshow(enh); ax1.axis('off')
        if col==0: ax1.set_ylabel('UNLET Enhanced',color='#22c55e',fontsize=11,fontweight='bold',rotation=90)
    cap.release()
    plt.tight_layout(pad=0.2)
    if save_path:
        plt.savefig(save_path,dpi=150,bbox_inches='tight',facecolor='#0a0f1e')
        print(f'Saved: {save_path}')
    plt.show()

show_video_comparison(
    CMP_OUTPUT,
    save_path=os.path.join(OUTPUT_DIR,'video_frames.png'))

In [ ]:
# ============================================================
# CELL 11 — Training Curves + Final Summary
# ============================================================

# Load history
hist_path = os.path.join(SAVE_DIR,'history.json')
with open(hist_path) as f:
    history = json.load(f)

ep=range(1,len(history['train'])+1)
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.patch.set_facecolor('#0f172a')
for ax in axes:
    ax.set_facecolor('#1e293b'); ax.tick_params(colors='white')
    ax.spines[['top','right','left','bottom']].set_color('#334155')

axes[0].plot(ep,history['train'],color='#38bdf8',lw=2,label='Train')
axes[0].plot(ep,history['val'],  color='#f59e0b',lw=2,ls='--',label='Val')
axes[0].set_title('Training Loss',color='white',fontsize=12,fontweight='bold')
axes[0].set_xlabel('Epoch',color='white'); axes[0].set_ylabel('Loss',color='white')
axes[0].legend(facecolor='#1e293b',labelcolor='white')

axes[1].plot(ep,history['psnr'],color='#22c55e',lw=2,label='PSNR (dB)')
ax2=axes[1].twinx()
ax2.plot(ep,history['ssim'],color='#f59e0b',lw=2,ls='--',label='SSIM')
ax2.tick_params(colors='white'); ax2.set_ylabel('SSIM',color='white')
axes[1].set_title('PSNR & SSIM',color='white',fontsize=12,fontweight='bold')
axes[1].set_xlabel('Epoch',color='white'); axes[1].set_ylabel('PSNR (dB)',color='white')
axes[1].legend(loc='lower right',facecolor='#1e293b',labelcolor='white')
ax2.legend(loc='center right',facecolor='#1e293b',labelcolor='white')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'training_curves.png'),
            dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()

# Final summary
print('\n'+'='*55)
print('  UNLET-ADAS FINAL SUMMARY')
print('='*55)
print(f'  Model      : Zero-DCE++ + CBAM Attention')
print(f'  Dataset    : LOL (485 train / 15 test) '
          f'+ {len(train_ds)-485 if len(train_ds)>485 else 0} extra unpaired')
print(f'  Epochs     : {len(history["train"])}')
print(f'  Best PSNR  : {max(history["psnr"]):.2f} dB')
print(f'  Best SSIM  : {max(history["ssim"]):.4f}')
print(f'  GitHub     : github.com/DEEK-SHITH/UNLET-ADAS')
print(f'  Demo       : unlet-adas-g4xvfhrfamxaqhpfuaqtri.streamlit.app')
print('='*55)
print('\nOutput files in Drive:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    sz=os.path.getsize(os.path.join(OUTPUT_DIR,fname))/(1024*1024)
    print(f'  {fname:<45} {sz:6.1f} MB')
print('='*55)